### Textual Analysis Pipeline Execution

This notebook serves as the execution interface for the textual analysis pipeline.
It imports the modular scripts developed for the project, including functions for
transcript processing, metadata extraction, and section parsing. The notebook first
adds the `program/scripts` directory to the Python path to ensure the custom modules
can be imported. It then loads the main pipeline functions required to process the
earnings call transcripts and generate the final dataset containing AI term counts,
LM sentiment metrics, and transcript metadata.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
SCRIPTS_DIR = PROJECT_ROOT / "program" / "scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

Import scripts

In [2]:
from pipeline import run_corpus, process_transcript
from metadata import extract_call_date, extract_fiscal_quarter_year
from text_processing import split_presentation_and_qa

Execute codes

In [3]:
df = run_corpus(
    input_root="../../data/raw/transcript",
    out_csv="../../data/processed/textual_analysis_results.csv",
    out_xlsx="../../data/processed/textual_analysis_results.xlsx",
    lm_dict_path="../../data/raw/Loughran-McDonald_MasterDictionary_1993-2024.csv",
)

df.head()

,company,ticker,date,file,total_words,pres_words,qa_words,core_hits_total,core_hits_pres,core_hits_qa,...,lm_uncertainty_qa,fiscal_quarter,fiscal_year,fiscal_period,core_per_1000_total,core_per_1000_pres,core_per_1000_qa,adj_per_1000_total,adj_per_1000_pres,adj_per_1000_qa
0,3M Co,MMM,2022-04-26,2022-Apr-26-MMM.N-140622159458-Transcript.txt,11615,5274,6153,0,0,0,...,0.012354,1,2022,Q1 2022,0.0,0.0,0.0,0.774860,1.706485,0.000000
1,3M Co,MMM,2022-01-25,2022-Jan-25-MMM.N-140663487123-Transcript.txt,11937,4250,7495,0,0,0,...,0.009925,4,2021,Q4 2021,0.0,0.0,0.0,0.753958,0.941176,0.667111
2,3M Co,MMM,2022-07-26,2022-Jul-26-MMM.N-140270570993-Transcript.txt,13203,5024,7976,0,0,0,...,0.011493,2,2022,Q2 2022,0.0,0.0,0.0,0.605923,1.393312,0.125376
3,3M Co,MMM,2022-10-25,2022-Oct-25-MMM.N-140586063160-Transcript.txt,10815,4276,6369,0,0,0,...,0.009781,3,2022,Q3 2022,0.0,0.0,0.0,2.311604,1.637044,2.826189
4,3M Co,MMM,2023-04-25,2023-Apr-25-MMM.N-136949071306-Transcript.txt,10820,3806,6832,0,0,0,...,0.009125,1,2023,Q1 2023,0.0,0.0,0.0,0.831793,1.576458,0.439110


Some companies change names during the period, and possibly the textual analysis engine would inaccruately capture these companies' names and tickers. Hence, we cross check with the `linking_table` to make sure all companies identified by the engine is linkable to a PERMNO number on the `linking_table`.

In [4]:
import pandas as pd

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

text_df = pd.read_csv(PROCESSED_DATA / "textual_analysis_results.csv")
link_df = pd.read_csv(RAW_DATA / "linking_table.csv")

# ---- textual analysis output ----
TEXT_COMPANY_COL = "company"
TEXT_TICKER_COL = "ticker"

# ---- linking table ----
LINK_COMPANY_COL = "IssuerNm"
LINK_PERMNO_COL = "PERMNO"
LINK_TICKER_COL = "Ticker"

# -------------------------------------------------
# 1. NORMALIZE (lowercase + strip)
# -------------------------------------------------
text_df["ticker_norm"] = text_df[TEXT_TICKER_COL].str.lower().str.strip()
link_df["ticker_norm"] = link_df[LINK_TICKER_COL].str.lower().str.strip()

text_df["company_norm"] = text_df[TEXT_COMPANY_COL].str.lower().str.strip()
link_df["company_norm"] = link_df[LINK_COMPANY_COL].str.lower().str.strip()

# -------------------------------------------------
# 2. UNIQUE VALUES
# -------------------------------------------------
text_tickers = set(text_df["ticker_norm"].dropna().unique())
link_tickers = set(link_df["ticker_norm"].dropna().unique())

text_companies = set(text_df["company_norm"].dropna().unique())
link_companies = set(link_df["company_norm"].dropna().unique())

# -------------------------------------------------
# 3. FIND MISSING
# -------------------------------------------------
missing_tickers = text_tickers - link_tickers
missing_companies = text_companies - link_companies

# -------------------------------------------------
# 4. PRINT RESULTS
# -------------------------------------------------
print(f"Missing tickers: {len(missing_tickers)}")
print(missing_tickers)

print("\nMissing companies: ", len(missing_companies))
print(missing_companies)

Missing tickers: 0
set()

Missing companies:  29
{'bristol-myers squibb co', 'coca-cola co', 'morgan stanley', 'chevron corp', 'us bancorp', 'amazon.com inc', 'mcdonald_s corp', 'blackrock inc', 'american tower corp', 'accenture plc', 'linde plc', 'at&t inc', 'merck & co inc', 'eli lilly and co', 'colgate-palmolive co', 'jpmorgan chase and co', 'comcast corp', 'international business machines corp', 'costco wholesale corp', 'proctor & gamble co', 'lowe_s companies inc', 't-mobile us ince', 'rtx corp', 'philips morris international inc', 'wells fargo & co', 'cvs health corp', 'walt disney co', 'duke energy corp', 'simon property group inc'}


Final inspection / Sanity checks

In [5]:
# ================================
# Final Inspection / Sanity Checks
# ================================

import pandas as pd

issues = []

# 1. LM tone should be between -1 and 1
lm_tone_cols = ["lm_tone_total", "lm_tone_pres", "lm_tone_qa"]

for col in lm_tone_cols:
    bad = df[(df[col] < -1) | (df[col] > 1)]
    if not bad.empty:
        print(f"\n⚠️ LM tone out of bounds in {col}:")
        display(bad[["company","ticker","file_name",col]])
        issues.append(col)

# 2. LM uncertainty should be between 0 and 1
lm_unc_cols = ["lm_uncertainty_total", "lm_uncertainty_pres", "lm_uncertainty_qa"]

for col in lm_unc_cols:
    bad = df[(df[col] < 0) | (df[col] > 1)]
    if not bad.empty:
        print(f"\n⚠️ LM uncertainty out of bounds in {col}:")
        display(bad[["company","ticker","file_name",col]])
        issues.append(col)

# 3. Presentation section not detected (0 words)
bad_pres = df[df["pres_words"] == 0]

if not bad_pres.empty:
    print("\n⚠️ Presentation section not detected (0 words):")
    display(bad_pres[["company","ticker","file_name","words_pres"]])
    issues.append("presentation_zero")

# 4. Q&A section not detected (0 words)
bad_qa = df[df["qa_words"] == 0]

if not bad_qa.empty:
    print("\n⚠️ Q&A section not detected (0 words):")
    display(bad_qa[["company","ticker","file_name","words_qa"]])
    issues.append("qa_zero")

# Final summary
if not issues:
    print("✅ All sanity checks passed. No anomalies detected.")
else:
    print(f"\nInspection completed. Issues detected in: {set(issues)}")

✅ All sanity checks passed. No anomalies detected.


In [6]:
# ============================================================
# Sanity Check: Inspect rows where fiscal_quarter / fiscal_year is NA
#
# Two possible root causes:
#   (A) ENGINE ISSUE  — the pattern IS in the transcript but the regex
#       missed it (e.g. format "First Quarter 2023", "1Q23", no space, etc.)
#   (B) FILE ISSUE    — the transcript genuinely does not mention the
#       fiscal period in a parseable form anywhere in the header
#
# Strategy:
#   1. Identify NA rows in the processed output
#   2. For each affected transcript file, scan the first 150 lines and
#      search for ANY quarter-like string to distinguish (A) from (B)
#   3. Print a diagnosis per file so you know where to act
# ============================================================

import pandas as pd
import re
from pathlib import Path

PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
TRANSCRIPT_DIR = PROJECT_ROOT / "data" / "raw" / "transcript"

df_check = pd.read_csv(PROCESSED_DATA / "textual_analysis_results.csv")

na_rows = df_check[df_check["fiscal_quarter"].isna() | df_check["fiscal_year"].isna()].copy()

print(f"Rows with missing fiscal_quarter or fiscal_year: {len(na_rows)} / {len(df_check)} total")
print(f"Affected companies: {sorted(na_rows['ticker'].unique())}\n")

if na_rows.empty:
    print("✅ No NA rows found — fiscal metadata is complete.")
else:
    # Patterns to search for manually in the transcript header
    # These are intentionally broader than the engine's regex to catch near-misses
    BROAD_PATTERNS = [
        (re.compile(r"\bQ[1-4]\s*20\d{2}\b",         re.IGNORECASE), "Standard  (Q3 2023)"),
        (re.compile(r"\b[1-4]Q\s*20\d{2}\b",         re.IGNORECASE), "Reversed  (3Q2023)"),
        (re.compile(r"\b(first|second|third|fourth)\s+quarter\b", re.IGNORECASE), "Written   (Third Quarter)"),
        (re.compile(r"\bfiscal\s+(quarter|q)\s*[1-4]\b", re.IGNORECASE), "FiscalQ   (Fiscal Q3)"),
        (re.compile(r"\bQ[1-4]\s*'?\d{2}\b",         re.IGNORECASE), "Short year (Q3'23)"),
    ]

    engine_issue   = []  # file has a quarter pattern the engine missed
    file_issue     = []  # file genuinely has no readable fiscal quarter info
    file_not_found = []  # transcript file couldn't be located on disk

    for _, row in na_rows.iterrows():
        fname = row["file"]
        fpath = TRANSCRIPT_DIR / fname

        if not fpath.exists():
            file_not_found.append(fname)
            continue

        # Read first 150 lines (wider than engine's 120 for a fairer check)
        with open(fpath, "r", encoding="utf-8", errors="replace") as f:
            head_lines = [f.readline() for _ in range(150)]
        head_text = "".join(head_lines)

        matched_patterns = []
        for pat, label in BROAD_PATTERNS:
            m = pat.search(head_text)
            if m:
                matched_patterns.append(f"{label} → '{m.group(0).strip()}'")

        if matched_patterns:
            engine_issue.append((row["ticker"], fname, matched_patterns))
        else:
            file_issue.append((row["ticker"], fname))

    # ── Report ──────────────────────────────────────────────────────────────
    print("=" * 70)
    print(f"  (A) ENGINE ISSUE  — pattern present but regex missed it : {len(engine_issue)}")
    print(f"  (B) FILE ISSUE    — no fiscal quarter found in header   : {len(file_issue)}")
    print(f"  (C) FILE NOT FOUND on disk                              : {len(file_not_found)}")
    print("=" * 70)

    if engine_issue:
        print("\n── (A) ENGINE ISSUES ──────────────────────────────────────────────")
        for ticker, fname, patterns in engine_issue:
            print(f"\n  [{ticker}]  {fname}")
            for p in patterns:
                print(f"    ↳ {p}")
        print(
            "\n  ⚙️  Fix: update FISCAL_Q_RE in metadata.py to cover these formats,"
            "\n     or add a second-pass pattern after the main one."
        )

    if file_issue:
        print("\n── (B) FILE ISSUES ────────────────────────────────────────────────")
        for ticker, fname in file_issue:
            print(f"  [{ticker}]  {fname}")
        print(
            "\n  📄  These transcripts have no quarter info in the first 150 lines."
            "\n     Options: (1) inspect files manually, (2) infer quarter from call date,"
            "\n     (3) accept NAs and exclude from fiscal-period-based analyses."
        )

    if file_not_found:
        print("\n── (C) FILES NOT FOUND ────────────────────────────────────────────")
        for fname in file_not_found:
            print(f"  {fname}")
        print("\n  ⚠️  These files are in the CSV but missing from the transcript folder.")


Rows with missing fiscal_quarter or fiscal_year: 0 / 1547 total
Affected companies: []

✅ No NA rows found — fiscal metadata is complete.
